# Chapter 5: PPO — Proximal Policy Optimization for Safe Learning

> **Book:** Agentic AI Engineering  
> **Part:** 2 — RL Methods  
> **System:** Guardian Angel System (GAS) — Multi-agent diabetic patient supervision

In [ ]:
# Install dependencies
# pip install gymnasium numpy matplotlib torch

In [ ]:
import os
TEST_MODE = os.getenv("NOTEBOOK_TEST_MODE", "0") == "1"
print(f"TEST_MODE: {TEST_MODE}")

---
## 🎯 1. Chapter Goal

By the end of this notebook you will:
- Understand **PPO's clipped objective** and why it prevents catastrophic updates
- Implement **PPO from scratch** on CartPole
- Visualize **policy loss, value loss, and KL divergence** over training
- Understand how PPO applies to **LLM token generation**
- See the effect of **clip_epsilon** hyperparameter sensitivity

---
## 📖 2. Theory Recap

### 2.1 Policy Gradient

The policy gradient theorem:
$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ \sum_t \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot A_t \right]$$

Where $A_t$ is the **advantage** (how much better was this action than average?).

**Problem:** Large gradient steps can destroy the policy. TRPO/PPO fix this.

### 2.2 PPO Clipped Objective

$$L^{CLIP}(\theta) = \mathbb{E}_t \left[ \min\left( r_t(\theta) A_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) A_t \right) \right]$$

Where $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)}$ is the **probability ratio**.

The clip prevents the ratio from going outside $[1-\epsilon, 1+\epsilon]$, limiting how much the policy can change per update.

### 2.3 Generalized Advantage Estimation (GAE)

$$A_t^{GAE} = \sum_{l=0}^{\infty} (\gamma \lambda)^l \delta_{t+l}$$

Where $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$ is the TD error.

- λ=0: pure TD (low variance, high bias)
- λ=1: pure MC (high variance, low bias)
- λ=0.95: good balance

### 2.4 Actor-Critic Architecture

```
State → Shared Encoder → Actor Head → π(a|s) (policy)
                       → Critic Head → V(s) (value function)
```

### 2.5 PPO for LLMs

In RLHF with PPO:
- **Actor**: The LLM being trained (generates tokens)
- **Critic**: A value head on top of the LLM
- **Reward**: From a reward model (human preferences)
- **Reference model**: The SFT model (KL penalty prevents drift)

$$r_{total} = r_{reward} - \beta \cdot KL(\pi_{\theta} || \pi_{ref})$$

---
## 🔨 3. Build It

### 3.1 PPO from Scratch on CartPole

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import matplotlib.pyplot as plt
from collections import deque

# ── Actor-Critic Network ────────────────────────────────────────────────────

class ActorCritic(nn.Module):
    """Shared encoder with separate actor and critic heads."""
    
    def __init__(self, state_dim, action_dim, hidden_dim=64):
        super().__init__()
        
        # Shared encoder
        self.encoder = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
        )
        
        # Actor head: outputs action probabilities
        self.actor = nn.Linear(hidden_dim, action_dim)
        
        # Critic head: outputs state value
        self.critic = nn.Linear(hidden_dim, 1)
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
                nn.init.zeros_(m.bias)
    
    def forward(self, state):
        features = self.encoder(state)
        action_logits = self.actor(features)
        value = self.critic(features)
        return action_logits, value
    
    def get_action(self, state):
        """Sample action and return log probability."""
        action_logits, value = self(state)
        dist = Categorical(logits=action_logits)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        return action.item(), log_prob, value.squeeze()
    
    def evaluate(self, states, actions):
        """Evaluate actions for PPO update."""
        action_logits, values = self(states)
        dist = Categorical(logits=action_logits)
        log_probs = dist.log_prob(actions)
        entropy = dist.entropy()
        return log_probs, values.squeeze(), entropy


# ── PPO Agent ───────────────────────────────────────────────────────────────

class PPOAgent:
    """PPO agent with clipped objective."""
    
    def __init__(
        self,
        state_dim,
        action_dim,
        lr=3e-4,
        gamma=0.99,
        gae_lambda=0.95,
        clip_epsilon=0.2,
        value_coef=0.5,
        entropy_coef=0.01,
        n_epochs=4,
        batch_size=64,
    ):
        self.gamma = gamma
        self.gae_lambda = gae_lambda
        self.clip_epsilon = clip_epsilon
        self.value_coef = value_coef
        self.entropy_coef = entropy_coef
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        
        self.network = ActorCritic(state_dim, action_dim)
        self.optimizer = optim.Adam(self.network.parameters(), lr=lr)
        
        # Rollout buffer
        self.reset_buffer()
    
    def reset_buffer(self):
        self.states = []
        self.actions = []
        self.log_probs = []
        self.rewards = []
        self.values = []
        self.dones = []
    
    def store(self, state, action, log_prob, reward, value, done):
        self.states.append(state)
        self.actions.append(action)
        self.log_probs.append(log_prob)
        self.rewards.append(reward)
        self.values.append(value)
        self.dones.append(done)
    
    def compute_gae(self, next_value):
        """Compute Generalized Advantage Estimation."""
        advantages = []
        gae = 0
        
        values = self.values + [next_value]
        
        for t in reversed(range(len(self.rewards))):
            delta = (self.rewards[t] + 
                     self.gamma * values[t+1] * (1 - self.dones[t]) - 
                     values[t])
            gae = delta + self.gamma * self.gae_lambda * (1 - self.dones[t]) * gae
            advantages.insert(0, gae)
        
        advantages = torch.tensor(advantages, dtype=torch.float32)
        returns = advantages + torch.tensor(self.values, dtype=torch.float32)
        
        # Normalize advantages
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        
        return advantages, returns
    
    def update(self, next_value):
        """PPO update step."""
        advantages, returns = self.compute_gae(next_value)
        
        states = torch.tensor(np.array(self.states), dtype=torch.float32)
        actions = torch.tensor(self.actions, dtype=torch.long)
        old_log_probs = torch.tensor([lp.item() for lp in self.log_probs], dtype=torch.float32)
        
        metrics = {'policy_loss': [], 'value_loss': [], 'entropy': [], 'kl_div': [], 'clip_frac': []}
        
        n = len(states)
        
        for epoch in range(self.n_epochs):
            # Mini-batch updates
            indices = torch.randperm(n)
            
            for start in range(0, n, self.batch_size):
                idx = indices[start:start + self.batch_size]
                
                batch_states = states[idx]
                batch_actions = actions[idx]
                batch_old_log_probs = old_log_probs[idx]
                batch_advantages = advantages[idx]
                batch_returns = returns[idx]
                
                # Evaluate current policy
                new_log_probs, values, entropy = self.network.evaluate(batch_states, batch_actions)
                
                # Probability ratio
                ratio = torch.exp(new_log_probs - batch_old_log_probs)
                
                # Clipped surrogate objective
                surr1 = ratio * batch_advantages
                surr2 = torch.clamp(ratio, 1 - self.clip_epsilon, 1 + self.clip_epsilon) * batch_advantages
                policy_loss = -torch.min(surr1, surr2).mean()
                
                # Value loss
                value_loss = nn.MSELoss()(values, batch_returns)
                
                # Entropy bonus (encourages exploration)
                entropy_loss = -entropy.mean()
                
                # Total loss
                total_loss = (policy_loss + 
                              self.value_coef * value_loss + 
                              self.entropy_coef * entropy_loss)
                
                self.optimizer.zero_grad()
                total_loss.backward()
                nn.utils.clip_grad_norm_(self.network.parameters(), 0.5)
                self.optimizer.step()
                
                # Track metrics
                with torch.no_grad():
                    kl = (batch_old_log_probs - new_log_probs).mean().item()
                    clip_frac = ((ratio - 1).abs() > self.clip_epsilon).float().mean().item()
                
                metrics['policy_loss'].append(policy_loss.item())
                metrics['value_loss'].append(value_loss.item())
                metrics['entropy'].append(entropy.mean().item())
                metrics['kl_div'].append(kl)
                metrics['clip_frac'].append(clip_frac)
        
        self.reset_buffer()
        return {k: np.mean(v) for k, v in metrics.items()}


print("PPO implementation ready!")
print(f"ActorCritic parameters: {sum(p.numel() for p in ActorCritic(4, 2).parameters()):,}")

In [ ]:
import gymnasium as gym

def train_ppo(env_name='CartPole-v1', n_steps=2048, n_updates=100, clip_epsilon=0.2, seed=42):
    """Train PPO agent and return metrics."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    env = gym.make(env_name)
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n
    
    agent = PPOAgent(state_dim, action_dim, clip_epsilon=clip_epsilon)
    
    all_rewards = []
    all_metrics = []
    episode_rewards = deque(maxlen=20)
    
    state, _ = env.reset(seed=seed)
    episode_reward = 0
    
    for update in range(n_updates):
        # Collect rollout
        for step in range(n_steps):
            state_tensor = torch.tensor(state, dtype=torch.float32)
            action, log_prob, value = agent.network.get_action(state_tensor)
            
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            agent.store(state, action, log_prob, reward, value.item(), done)
            
            state = next_state
            episode_reward += reward
            
            if done:
                episode_rewards.append(episode_reward)
                all_rewards.append(episode_reward)
                episode_reward = 0
                state, _ = env.reset()
        
        # Compute next value for GAE
        with torch.no_grad():
            _, next_value = agent.network(torch.tensor(state, dtype=torch.float32))
        
        # PPO update
        metrics = agent.update(next_value.item())
        all_metrics.append(metrics)
        
        if (update + 1) % 10 == 0:
            avg_reward = np.mean(episode_rewards) if episode_rewards else 0
            print(f"Update {update+1:3d}/{n_updates} | Avg Reward: {avg_reward:6.1f} | "
                  f"Policy Loss: {metrics['policy_loss']:6.3f} | "
                  f"KL: {metrics['kl_div']:6.4f} | "
                  f"Clip Frac: {metrics['clip_frac']:.2f}")
    
    env.close()
    return all_rewards, all_metrics


n_updates = 50 if not TEST_MODE else 10
print(f"Training PPO on CartPole-v1 ({n_updates} updates)...")
print("=" * 70)
rewards, metrics = train_ppo(n_updates=n_updates)

In [ ]:
# Plot PPO training metrics
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Episode rewards
ax1 = axes[0, 0]
window = 20
ax1.plot(rewards, alpha=0.3, color='blue')
if len(rewards) >= window:
    smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
    ax1.plot(range(window-1, len(rewards)), smoothed, 'b-', linewidth=2)
ax1.axhline(y=475, color='red', linestyle='--', label='Solved (475)')
ax1.set_xlabel('Episode')
ax1.set_ylabel('Episode Reward')
ax1.set_title('PPO: Episode Rewards', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Policy loss
ax2 = axes[0, 1]
policy_losses = [m['policy_loss'] for m in metrics]
ax2.plot(policy_losses, 'r-', linewidth=2)
ax2.set_xlabel('Update')
ax2.set_ylabel('Policy Loss')
ax2.set_title('PPO: Policy Loss', fontweight='bold')
ax2.grid(True, alpha=0.3)

# Value loss
ax3 = axes[0, 2]
value_losses = [m['value_loss'] for m in metrics]
ax3.plot(value_losses, 'g-', linewidth=2)
ax3.set_xlabel('Update')
ax3.set_ylabel('Value Loss')
ax3.set_title('PPO: Value Loss (Critic)', fontweight='bold')
ax3.grid(True, alpha=0.3)

# KL divergence
ax4 = axes[1, 0]
kl_divs = [m['kl_div'] for m in metrics]
ax4.plot(kl_divs, 'm-', linewidth=2)
ax4.axhline(y=0.02, color='red', linestyle='--', label='KL target (0.02)')
ax4.set_xlabel('Update')
ax4.set_ylabel('KL Divergence')
ax4.set_title('PPO: KL Divergence\n(policy change per update)', fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3)

# Clip fraction
ax5 = axes[1, 1]
clip_fracs = [m['clip_frac'] for m in metrics]
ax5.plot(clip_fracs, 'orange', linewidth=2)
ax5.axhline(y=0.1, color='red', linestyle='--', label='Target (~10%)')
ax5.set_xlabel('Update')
ax5.set_ylabel('Clip Fraction')
ax5.set_title('PPO: Clip Fraction\n(% of clipped ratios)', fontweight='bold')
ax5.legend()
ax5.grid(True, alpha=0.3)

# Entropy
ax6 = axes[1, 2]
entropies = [m['entropy'] for m in metrics]
ax6.plot(entropies, 'cyan', linewidth=2)
ax6.set_xlabel('Update')
ax6.set_ylabel('Policy Entropy')
ax6.set_title('PPO: Policy Entropy\n(exploration level)', fontweight='bold')
ax6.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ppo_training_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.2 Clip Epsilon Sensitivity Analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Visualize the clipped objective for different epsilon values
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

ratios = np.linspace(0.5, 1.5, 200)
advantage = 1.0  # positive advantage

for ax, epsilon, title in zip(axes, [0.05, 0.2, 0.5], 
                               ['ε=0.05 (too conservative)', 'ε=0.2 (standard)', 'ε=0.5 (too aggressive)']):
    # Unclipped objective
    unclipped = ratios * advantage
    
    # Clipped objective
    clipped_ratio = np.clip(ratios, 1 - epsilon, 1 + epsilon)
    clipped = clipped_ratio * advantage
    
    # PPO objective (min of both)
    ppo_obj = np.minimum(unclipped, clipped)
    
    ax.plot(ratios, unclipped, 'b--', label='Unclipped', linewidth=2, alpha=0.7)
    ax.plot(ratios, clipped, 'r--', label='Clipped', linewidth=2, alpha=0.7)
    ax.plot(ratios, ppo_obj, 'g-', label='PPO (min)', linewidth=3)
    ax.axvline(x=1-epsilon, color='gray', linestyle=':', alpha=0.7)
    ax.axvline(x=1+epsilon, color='gray', linestyle=':', alpha=0.7)
    ax.axvline(x=1.0, color='black', linestyle='-', alpha=0.3, label='r=1 (no change)')
    ax.fill_betweenx([0, 1.5], 1-epsilon, 1+epsilon, alpha=0.1, color='green', label='Allowed region')
    ax.set_xlabel('Probability Ratio r(θ)')
    ax.set_ylabel('Objective Value')
    ax.set_title(f'PPO Clipped Objective\n{title}', fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1.6)

plt.tight_layout()
plt.savefig('ppo_clip_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

print("Clip Epsilon Analysis:")
print("  ε=0.05: Very conservative — slow learning, stable but may not converge")
print("  ε=0.2:  Standard — good balance of stability and learning speed")
print("  ε=0.5:  Too aggressive — large policy changes, unstable training")
print("\nFor GAS Safety Critic: use ε=0.1-0.2 (medical safety requires stability)")

### 3.3 PPO for LLM Token Generation (Conceptual)

In [ ]:
# Conceptual demonstration of PPO for LLMs
# (Pseudocode + visualization — actual training requires TRL)

print("""
PPO for LLM Token Generation — Conceptual Flow
===============================================

SETUP:
  actor_model = SFT_model  (the model we're training)
  ref_model   = SFT_model  (frozen reference — prevents drift)
  reward_model = SafetyCritic  (scores responses)
  critic_head = ValueHead(actor_model)  (predicts expected reward)

ROLLOUT (collect data):
  for prompt in batch:
    response = actor_model.generate(prompt)  # sample tokens
    reward = reward_model(prompt, response)  # score the response
    kl_penalty = KL(actor_model(prompt) || ref_model(prompt))  # per token
    total_reward = reward - β * kl_penalty  # β controls KL weight

PPO UPDATE:
  for each token position t in response:
    ratio = exp(log_prob_new[t] - log_prob_old[t])
    advantage = GAE(rewards, values)  # from critic
    
    # Clipped objective
    L_clip = min(ratio * A_t, clip(ratio, 1-ε, 1+ε) * A_t)
    
    # Value loss
    L_value = MSE(critic(state), returns)
    
    # Total loss
    L = -L_clip + c1 * L_value - c2 * entropy
    
    optimizer.step(L)

KEY DIFFERENCES from CartPole PPO:
  1. State = (prompt + generated tokens so far)
  2. Action = next token (vocabulary size ~32K-128K)
  3. Episode = one complete response generation
  4. Reward = sparse (only at end of response)
  5. KL penalty prevents the model from drifting too far from SFT
""")

# Visualize the RLHF pipeline
fig, ax = plt.subplots(figsize=(14, 6))
ax.axis('off')

# Draw the pipeline
boxes = [
    (0.05, 0.5, 'Prompt\n"Glucose=280\nwhat to do?"', '#3498db'),
    (0.25, 0.7, 'Actor LLM\n(SFT model)\n→ generates response', '#2ecc71'),
    (0.25, 0.3, 'Reference LLM\n(frozen SFT)\n→ KL penalty', '#95a5a6'),
    (0.55, 0.5, 'Reward Model\n(Safety Critic)\n→ safety score', '#e74c3c'),
    (0.75, 0.5, 'PPO Update\nclipped objective\n+ KL penalty', '#9b59b6'),
    (0.95, 0.5, 'Better\nActor LLM\n✅', '#f39c12'),
]

for x, y, text, color in boxes:
    ax.add_patch(plt.Rectangle((x-0.08, y-0.15), 0.16, 0.30, 
                                facecolor=color, alpha=0.3, edgecolor=color, linewidth=2))
    ax.text(x, y, text, ha='center', va='center', fontsize=9, fontweight='bold')

# Arrows
arrows = [
    (0.13, 0.5, 0.17, 0.7),   # prompt → actor
    (0.13, 0.5, 0.17, 0.3),   # prompt → reference
    (0.33, 0.7, 0.47, 0.55),  # actor → reward
    (0.33, 0.3, 0.47, 0.45),  # reference → reward (KL)
    (0.63, 0.5, 0.67, 0.5),   # reward → PPO
    (0.83, 0.5, 0.87, 0.5),   # PPO → better model
    (0.75, 0.35, 0.75, 0.2),  # PPO → back to actor (loop)
]

for x1, y1, x2, y2 in arrows:
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='black', lw=2))

ax.set_xlim(0, 1.05)
ax.set_ylim(0, 1)
ax.set_title('PPO for LLM Alignment (RLHF Pipeline)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('ppo_llm_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🏥 4. GAS Connection

PPO is used to train the GAS Safety Critic to give better safety scores. The clipped objective prevents the critic from changing too drastically — important for medical safety.

**Why PPO's stability matters for GAS:**

1. **Medical safety is non-negotiable**: A policy that suddenly starts recommending dangerous insulin doses (due to an unstable update) could harm patients. The clip prevents this.

2. **KL penalty keeps the model grounded**: The reference model (SFT) represents the "safe baseline" behavior. The KL penalty ensures the RL-trained model doesn't drift too far from this baseline.

3. **Asymmetric rewards work with PPO**: The -10 reward for hypoglycemia vs -2 for hyperglycemia creates a strong gradient signal that PPO can safely follow without overshooting.

**GAS PPO hyperparameters:**
```python
ppo_config = {
    'clip_epsilon': 0.1,    # conservative for medical safety
    'kl_coef': 0.1,         # strong KL penalty
    'gamma': 0.99,          # long-horizon (24-hour episodes)
    'gae_lambda': 0.95,     # standard
    'n_epochs': 4,          # standard
    'batch_size': 32,       # small batches for stability
}
```

---
## ✅ 5. Checkpoint

### What We Built

| Component | Description |
|-----------|-------------|
| `ActorCritic` | Shared encoder with actor/critic heads |
| `PPOAgent` | Full PPO with GAE, clipped objective, mini-batches |
| CartPole training | Demonstrated convergence |
| Clip sensitivity | Showed ε=0.05 vs 0.2 vs 0.5 |
| LLM PPO diagram | Conceptual RLHF pipeline |

### Key Takeaways

1. **The clip is the key innovation** — it prevents catastrophically large policy updates
2. **GAE balances bias-variance** — λ=0.95 is the standard choice
3. **KL divergence monitoring** — if KL > 0.02, the update was too large
4. **For LLMs**: PPO requires 4 models (actor, critic, reference, reward) — memory-intensive

### What Comes Next

**Chapter 6: DPO** — Direct Preference Optimization eliminates the need for a separate reward model and value network, making alignment much simpler and more memory-efficient.

```
Ch5 ✅ (PPO) → Ch6 (DPO — simpler alignment) → Ch7 (GRPO)
```